# 02 — Z-score vs Min-Max : correction du clampage du momentum

**Problème confirmé** : la normalisation min-max cross-sectionnelle sur 5 tickers clampe ~40 % des observations de `momentum_score` à exactement ±1. DIS reste collé à +0.50 de `global_score` quasi tout le mois, TGT à -0.50 — indépendamment de l'écart réel de performance entre eux.

**Ce notebook** :
1. Montre l'effet du clampage visuellement
2. Implémente une normalisation **z-score cross-sectionnel** (centré-réduit)
3. Option bonus : **tanh(z-score)** — garde un signal borné sans clampage dur
4. Compare les 3 normalisations avec un backtest complet sur mars–mai 2026
5. Produit la version corrigée de `signals.py` à copier dans le projet

In [ ]:
import sys
from pathlib import Path
# Chemin robuste : fonctionne si le notebook est lancé depuis
# backtesting_2/notebooks, backtesting_2, ou la racine du repo.
CWD = Path.cwd().resolve()
if (CWD / 'backtesting_2').exists():
    ROOT = CWD
elif CWD.name == 'notebooks' and CWD.parent.name == 'backtesting_2':
    ROOT = CWD.parent.parent
elif CWD.name == 'backtesting_2':
    ROOT = CWD.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from pypfopt import EfficientFrontier, expected_returns, risk_models

TICKERS = ['NKE', 'TGT', 'DIS', 'SBUX', 'TSLA']
DATA_INPUT  = ROOT / 'backtesting_2' / 'data_input'
DATA_OUTPUT = ROOT / 'backtesting_2' / 'data_output'
COLORS = {'NKE': '#1f77b4', 'TGT': '#ff7f0e', 'DIS': '#2ca02c', 'SBUX': '#d62728', 'TSLA': '#9467bd'}
print("Imports OK")

## 1. Charger les scores momentum bruts (avant normalisation)

In [ ]:
mom_df = pd.read_csv(DATA_OUTPUT / 'daily_momentum_scores.csv', parse_dates=['date'])

# On a besoin du momentum_raw (non-normalisé) — on le recalcule à partir des colonnes mom_1m, mom_3m
mom_df['momentum_raw'] = mom_df[['mom_1m', 'mom_3m']].mean(axis=1)

raw_piv  = mom_df.pivot(index='date', columns='ticker', values='momentum_raw')[TICKERS]
norm_piv = mom_df.pivot(index='date', columns='ticker', values='momentum_score')[TICKERS]

print(f"Dates : {raw_piv.index[0].date()} → {raw_piv.index[-1].date()}, {len(raw_piv)} jours")
raw_piv.describe().round(4)

## 2. Trois normalisations côte à côte

In [ ]:
def normalize_minmax(raw_row: pd.Series) -> pd.Series:
    """Normalisation min-max cross-sectionnelle → [-1, +1]. ACTUELLE."""
    lo, hi = raw_row.min(), raw_row.max()
    if hi - lo < 1e-10:
        return pd.Series(0.0, index=raw_row.index)
    return 2.0 * (raw_row - lo) / (hi - lo) - 1.0

def normalize_zscore(raw_row: pd.Series) -> pd.Series:
    """Z-score cross-sectionnel. Reflète l'écart réel en σ."""
    mu, sigma = raw_row.mean(), raw_row.std()
    if sigma < 1e-10:
        return pd.Series(0.0, index=raw_row.index)
    return (raw_row - mu) / sigma

def normalize_tanh_zscore(raw_row: pd.Series, scale: float = 1.0) -> pd.Series:
    """tanh(z-score) → borné dans (-1, +1) SANS clampage dur."""
    z = normalize_zscore(raw_row)
    return np.tanh(z * scale)

# Appliquer jour par jour
minmax_piv  = raw_piv.apply(normalize_minmax,       axis=1)
zscore_piv  = raw_piv.apply(normalize_zscore,       axis=1)
tanh_piv    = raw_piv.apply(normalize_tanh_zscore,  axis=1)

# Taux de clampage
for label, piv in [('Min-Max (actuel)', minmax_piv), ('Z-score', zscore_piv), ('tanh(Z-score)', tanh_piv)]:
    vals = piv.values.flatten()
    c = np.sum(np.abs(vals) >= 0.999)
    print(f"{label:20s} : {100*c/len(vals):.1f}% des observations à |score| ≥ 0.999")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

pairs = [
    ('Min-Max cross-sectionnel (ACTUEL)', minmax_piv),
    ('Z-score cross-sectionnel (PROPOSÉ)', zscore_piv),
    ('tanh(Z-score) — borné sans clampage dur', tanh_piv),
]

for ax, (title, piv) in zip(axes, pairs):
    for t in TICKERS:
        ax.plot(piv.index, piv[t], label=t, color=COLORS[t], alpha=0.85)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(title)
    ax.set_ylabel('Score normalisé')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.2)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
fig.suptitle('Comparaison des normalisations du momentum cross-sectionnel', fontsize=13, y=1.01)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Backtest complet — 3 variantes de normalisation

On rejoue la boucle de backtest simplifiée (sans FinBERT pour la rapidité) pour comparer les 3 normalisations sur mars–mai 2026.

In [ ]:
# Téléchargement prix
PRICE_START = '2025-10-01'
BACKTEST_START = '2026-03-02'
BACKTEST_END   = '2026-05-29'

prices = yf.download(TICKERS, start=PRICE_START, end=BACKTEST_END,
                     auto_adjust=True, progress=False)['Close']
prices.columns = [c if isinstance(c, str) else c[0] for c in prices.columns]
prices = prices[TICKERS]

backtest_days = pd.bdate_range(BACKTEST_START, BACKTEST_END)
backtest_days = [d for d in backtest_days if d in prices.index]
print(f"Prix : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Jours de backtest : {len(backtest_days)}")

In [ ]:
def run_backtest_variant(norm_func, label,
                         SIGNAL_STRENGTH=0.20, RISK_AVERSION=1.0,
                         WEIGHT_BOUNDS=(0.05, 0.40), MOMENTUM_WINDOWS=[1, 3]):
    """
    Boucle de backtest minimaliste — utilise uniquement le momentum (pas FinBERT)
    pour tester l'effet de la normalisation seule.
    """
    TRADING_DAYS_PER_MONTH = 21
    capital = 100_000.0
    portfolio_values = [capital]
    weights_log = []

    for day in backtest_days[:-1]:  # dernier jour : pas de J+1
        train = prices.loc[:day].dropna(how='all')
        if len(train) < 65:
            weights_log.append({t: 1/len(TICKERS) for t in TICKERS})
            next_day = backtest_days[backtest_days.index(day) + 1]
            ret = prices.loc[next_day] / prices.loc[day] - 1
            portfolio_values.append(portfolio_values[-1] * (1 + sum(1/len(TICKERS) * ret[t] for t in TICKERS)))
            continue

        # Momentum raw
        raw_vals = {}
        for t in TICKERS:
            if t not in train.columns:
                raw_vals[t] = 0.0
                continue
            s = train[t].dropna()
            window_rets = []
            for w in MOMENTUM_WINDOWS:
                lag = w * TRADING_DAYS_PER_MONTH
                if len(s) > lag:
                    window_rets.append(float(s.iloc[-1]) / float(s.iloc[-(lag+1)]) - 1)
            raw_vals[t] = np.mean(window_rets) if window_rets else 0.0

        raw_series = pd.Series(raw_vals)
        norm_scores = norm_func(raw_series)  # normalisation à tester

        # mu_adjusted et covariance
        try:
            mu_hist = expected_returns.mean_historical_return(train, compounding=True, frequency=252)
            mu_adj  = mu_hist + SIGNAL_STRENGTH * norm_scores.reindex(mu_hist.index).fillna(0)
            S = risk_models.CovarianceShrinkage(train, frequency=252).ledoit_wolf()

            ef = EfficientFrontier(mu_adj, S, weight_bounds=WEIGHT_BOUNDS)
            ef.max_quadratic_utility(risk_aversion=RISK_AVERSION)
            w = dict(ef.clean_weights())
        except Exception:
            w = {t: 1/len(TICKERS) for t in TICKERS}

        weights_log.append(w)

        # Rendement J→J+1
        idx = backtest_days.index(day)
        next_day = backtest_days[idx + 1]
        port_ret = sum(w.get(t, 0) * (prices.loc[next_day, t] / prices.loc[day, t] - 1) for t in TICKERS)
        portfolio_values.append(portfolio_values[-1] * (1 + port_ret))

    return pd.Series(portfolio_values, index=backtest_days[:len(portfolio_values)]), pd.DataFrame(weights_log, index=backtest_days[:len(weights_log)])

print("Fonction de backtest définie.")

In [ ]:
print("Backtest en cours... (3 variantes × ~60 jours)")

vals_minmax, wts_minmax = run_backtest_variant(normalize_minmax,       'Min-Max')
print("  ✓ Min-Max terminé")
vals_zscore, wts_zscore = run_backtest_variant(normalize_zscore,       'Z-score')
print("  ✓ Z-score terminé")
vals_tanh,   wts_tanh   = run_backtest_variant(normalize_tanh_zscore,  'tanh(Z)')
print("  ✓ tanh(Z-score) terminé")

# Benchmark equal-weight
ew_ret = prices.loc[backtest_days].pct_change().fillna(0).mean(axis=1)
vals_ew = (1 + ew_ret).cumprod() * 100_000

print("Done.")

In [ ]:
def sharpe(series: pd.Series, freq=252) -> float:
    rets = series.pct_change().dropna()
    if rets.std() == 0:
        return 0.0
    return float(rets.mean() / rets.std() * np.sqrt(freq))

def max_dd(series: pd.Series) -> float:
    roll_max = series.cummax()
    return float(((series - roll_max) / roll_max).min())

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
variants = [
    ('Min-Max (actuel)',           vals_minmax, '#e74c3c'),
    ('Z-score (proposé)',          vals_zscore, '#2ecc71'),
    ('tanh(Z-score)',              vals_tanh,   '#3498db'),
    ('Equal-weight (benchmark)',   vals_ew,     '#f39c12'),
]

for label, vals, color in variants:
    ax.plot(vals.index, vals, label=label, color=color, lw=2)

ax.axhline(100_000, color='gray', lw=0.8, ls='--')
ax.set_title('Backtest mars–mai 2026 : effet de la normalisation du momentum')
ax.set_ylabel('Valeur du portefeuille ($)')
ax.legend()
ax.grid(alpha=0.2)

# Tableau des métriques
ax2 = axes[1]
ax2.axis('off')
table_data = []
for label, vals, _ in variants:
    total_ret = (vals.iloc[-1] / vals.iloc[0] - 1) * 100
    table_data.append([
        label,
        f"{total_ret:+.2f}%",
        f"{sharpe(vals):.2f}",
        f"{100*max_dd(vals):.2f}%",
        f"${vals.iloc[-1]:,.0f}"
    ])

tbl = ax2.table(
    cellText=table_data,
    colLabels=['Stratégie', 'Rendement', 'Sharpe', 'Max DD', 'Capital final'],
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2)
ax2.set_title('Métriques comparatives', pad=20)

plt.tight_layout()
plt.show()

print("\nConclusion :")
best = max(variants[:-1], key=lambda x: sharpe(x[1]))
print(f"  Meilleure normalisation (Sharpe) : {best[0]}")

## 4. Poids journaliers — visualisation de l'effet sur l'allocation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, wts, color) in zip(axes, [
    ('Min-Max (actuel)',  wts_minmax, None),
    ('Z-score',           wts_zscore, None),
    ('tanh(Z-score)',     wts_tanh,   None),
]):
    wts_filled = wts.fillna(0)
    bottom = np.zeros(len(wts_filled))
    for t in TICKERS:
        if t in wts_filled.columns:
            ax.bar(range(len(wts_filled)), wts_filled[t], bottom=bottom,
                   label=t, color=COLORS[t], width=0.8)
            bottom += wts_filled[t].values
    ax.set_title(label)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Poids')
    ax.set_xlabel('Jour de trading')

axes[-1].legend(loc='upper right')
fig.suptitle('Allocation journalière par normalisation', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Code à intégrer dans `signals.py`

Copie le bloc ci-dessous pour remplacer la normalisation min-max dans `signals.py` :

In [ ]:
code = '''
# Dans compute_daily_momentum(), remplacer le bloc de normalisation par :

# ── Normalisation z-score cross-sectionnel (remplace min-max) ────────────────
# Avantage : reflète l'écart réel en σ plutôt qu'un simple classement.
# Sur N=5 tickers, le min-max clampait ~40 % des observations à ±1 ;
# le z-score préserve la magnitude relative des écarts.
raw = df["momentum_raw"]
mu_cross   = raw.mean()
std_cross  = raw.std()
if std_cross < 1e-10:
    df["momentum_score"] = 0.0
else:
    z = (raw - mu_cross) / std_cross
    df["momentum_score"] = np.tanh(z)   # tanh → borné (-1,+1) sans clampage dur
    # Alternative pure z-score (non borné) : df["momentum_score"] = z

df["momentum_score"] = df["momentum_score"].round(6)
'''
print(code)